# Math Foundations for Deep Learning

**Linear algebra:** vectors, matrices, dot product, eigenvalues
**Calculus:** derivatives, gradients, the chain rule, and how they combine into backpropagation

The focus is intuition, with small numpy experiments and plots. The last linear-algebra section applies eigenvalues to `Training.csv` (the symptom dataset).

In [ ]:
%pip install -q numpy pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (5, 4)
plt.rcParams["axes.grid"] = True

# Part 1: Linear algebra

## 1.1 Vectors

A vector is a list of numbers, read either as a point or as an arrow from the origin. In machine learning it is usually a data point: one row of features.

In [ ]:
v = np.array([3.0, 1.0])
w = np.array([1.0, 2.0])

print("v + w =", v + w)
print("2 * v =", 2 * v)
print("shape:", v.shape)
print("length of v:", np.linalg.norm(v), "=", np.sqrt((v ** 2).sum()))

In [ ]:
def draw(vectors, colors, labels, origin=(0, 0)):
    for vec, c, l in zip(vectors, colors, labels):
        plt.quiver(*origin, *vec, angles="xy", scale_units="xy", scale=1, color=c, label=l)
    pts = np.array(vectors)
    lim = np.abs(pts).max() + 1
    plt.xlim(-1, lim)
    plt.ylim(-1, lim)
    plt.gca().set_aspect("equal")
    plt.legend()


draw([v, w, v + w], ["tab:blue", "tab:orange", "tab:green"], ["v", "w", "v + w"])
plt.title("Vector addition: tip to tail")
plt.show()

## 1.2 Dot product

The dot product multiplies matching entries and sums them. Geometrically it measures how much two vectors point the same way: large and positive when aligned, zero when perpendicular, negative when opposed.

$a \cdot b = \sum_i a_i b_i = \|a\|\|b\|\cos\theta$

In [ ]:
a = np.array([2.0, 1.0])
b = np.array([1.0, 3.0])

dot = a @ b
cos = dot / (np.linalg.norm(a) * np.linalg.norm(b))
print("dot:", dot)
print("cosine similarity:", cos)
print("angle (degrees):", np.degrees(np.arccos(cos)))

print("perpendicular dot:", np.array([1.0, 0.0]) @ np.array([0.0, 5.0]))
print("opposite dot:", np.array([1.0, 0.0]) @ np.array([-2.0, 0.0]))

In [ ]:
proj = (a @ b) / (b @ b) * b
draw([a, b, proj], ["tab:blue", "tab:orange", "tab:red"], ["a", "b", "projection of a on b"])
plt.plot([a[0], proj[0]], [a[1], proj[1]], "k--", linewidth=1)
plt.title("Projection: how much of a lies along b")
plt.show()

A neuron computes a dot product between its weights and the input, then adds a bias. The output is large when the input points in the direction the weights are looking for.

In [ ]:
weights = np.array([0.8, -0.5, 0.3])
x1 = np.array([1.0, 0.0, 1.0])
x2 = np.array([0.0, 1.0, 0.0])
bias = 0.1
print("neuron on x1:", weights @ x1 + bias)
print("neuron on x2:", weights @ x2 + bias)

## 1.3 Matrices as transformations

A matrix maps vectors to vectors. Its columns say where the basis vectors $(1,0)$ and $(0,1)$ land, and everything else follows by linearity. Matrix–vector multiplication is a batch of dot products, one per row.

In [ ]:
A = np.array([[2.0, 1.0],
              [0.0, 1.0]])
x = np.array([1.0, 2.0])

print("A @ x =", A @ x)
print("rows as dot products:", [A[0] @ x, A[1] @ x])
print("columns combined:", x[0] * A[:, 0] + x[1] * A[:, 1])

In [ ]:
def grid_lines(n=7, lim=3):
    t = np.linspace(-lim, lim, 50)
    lines = []
    for k in np.linspace(-lim, lim, n):
        lines.append(np.stack([t, np.full_like(t, k)]))
        lines.append(np.stack([np.full_like(t, k), t]))
    return lines


def show_transform(M, title):
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    for ax, mat, name in zip(axes, [np.eye(2), M], ["before", "after"]):
        for line in grid_lines():
            out = mat @ line
            ax.plot(out[0], out[1], color="lightsteelblue", linewidth=0.8)
        for col, c in zip(mat.T, ["tab:red", "tab:green"]):
            ax.quiver(0, 0, *col, angles="xy", scale_units="xy", scale=1, color=c)
        ax.set_xlim(-6, 6)
        ax.set_ylim(-6, 6)
        ax.set_aspect("equal")
        ax.set_title(f"{title}: {name}")
    plt.show()


theta = np.radians(40)
rotation = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
scaling = np.array([[2.0, 0.0], [0.0, 0.5]])
shear = np.array([[1.0, 1.0], [0.0, 1.0]])
collapse = np.array([[1.0, 1.0], [1.0, 1.0]])

show_transform(rotation, "rotation")
show_transform(scaling, "scaling")
show_transform(shear, "shear")
show_transform(collapse, "singular (rank 1)")

A rotation preserves lengths, a scaling stretches axes, a shear slants the grid, and a singular matrix flattens the plane onto a line, destroying information that cannot be recovered. Its determinant is 0.

In [ ]:
for name, M in [("rotation", rotation), ("scaling", scaling), ("shear", shear), ("singular", collapse)]:
    print(f"{name:9s} det = {np.linalg.det(M):6.3f}  rank = {np.linalg.matrix_rank(M)}")

### Composition and neural network layers

Applying one matrix and then another is the same as multiplying them first. Order matters. A linear layer is $y = Wx + b$, and a network without nonlinearities collapses to a single matrix.

In [ ]:
print("rotate then scale:\n", scaling @ rotation)
print("scale then rotate:\n", rotation @ scaling)
print("same?", np.allclose(scaling @ rotation, rotation @ scaling))

x = rng.normal(size=(4,))
W1 = rng.normal(size=(5, 4))
W2 = rng.normal(size=(3, 5))
print("two layers:", W2 @ (W1 @ x))
print("one merged:", (W2 @ W1) @ x)

In [ ]:
batch = rng.normal(size=(8, 4))
W = rng.normal(size=(4, 3))
b = np.zeros(3)
out = batch @ W + b
print(batch.shape, "@", W.shape, "->", out.shape)

relu = lambda z: np.maximum(0, z)
two_layer_nonlinear = relu(batch @ rng.normal(size=(4, 6))) @ rng.normal(size=(6, 3))
print(two_layer_nonlinear.shape)

## 1.4 Eigenvalues and eigenvectors

Most vectors change direction when a matrix acts on them. An **eigenvector** is a special direction that only gets stretched or flipped: $Av = \lambda v$. The factor $\lambda$ is its **eigenvalue**.

They reveal the axes along which a transformation acts as pure scaling.

In [ ]:
A = np.array([[3.0, 1.0],
              [1.0, 2.0]])
vals, vecs = np.linalg.eigh(A)
print("eigenvalues:", vals)
print("eigenvectors (columns):\n", vecs)

for lam, vec in zip(vals, vecs.T):
    print(f"A v = {A @ vec},  lambda v = {lam * vec}")

In [ ]:
angles = np.linspace(0, 2 * np.pi, 200)
circle = np.stack([np.cos(angles), np.sin(angles)])
ellipse = A @ circle

plt.plot(*circle, label="unit circle")
plt.plot(*ellipse, label="after A")
for lam, vec in zip(vals, vecs.T):
    plt.quiver(0, 0, *vec, angles="xy", scale_units="xy", scale=1, color="tab:red")
    plt.quiver(0, 0, *(lam * vec), angles="xy", scale_units="xy", scale=1, color="tab:green")
plt.gca().set_aspect("equal")
plt.legend()
plt.title("Eigenvectors are the ellipse's axes")
plt.show()

Red arrows are the eigenvectors and green arrows are the same directions after $A$ acts, stretched by $\lambda$. For a symmetric matrix like this one, the eigenvectors are perpendicular and form the principal axes of the stretched circle.

A matrix with a complex eigenvalue has no direction that stays on its line. A rotation is the classic example.

In [ ]:
print("rotation eigenvalues:", np.linalg.eigvals(rotation))
print("shear eigenvalues:", np.linalg.eigvals(shear))

### Power iteration

Repeatedly applying a matrix to a random vector makes it align with the eigenvector of the largest eigenvalue, because that direction grows fastest. This idea underlies PageRank and explains why repeated multiplication in deep networks makes signals explode or vanish.

In [ ]:
u = rng.normal(size=2)
for step in range(1, 11):
    u = A @ u
    u = u / np.linalg.norm(u)
    if step in (1, 2, 5, 10):
        print(f"step {step:2d}: {u}")

print("top eigenvector:", vecs[:, np.argmax(vals)])
print("Rayleigh estimate of lambda_max:", u @ A @ u, "vs", vals.max())

In [ ]:
for scale in [0.9, 1.0, 1.1]:
    M = scale * rotation
    z = np.array([1.0, 0.0])
    for _ in range(50):
        z = M @ z
    print(f"largest |eigenvalue| = {scale}: |z| after 50 applications = {np.linalg.norm(z):.4f}")

### Eigenvalues of data: PCA on the symptom dataset

The covariance matrix of the data is symmetric. Its eigenvectors are the directions of greatest variance, and each eigenvalue is the variance along that direction. This is PCA.

In [ ]:
df = pd.read_csv("Training.csv")
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
df.columns = df.columns.str.strip()
features = [c for c in df.columns if c != "prognosis"]

X = df[features].values.astype(float)
print(X.shape)

Xc = X - X.mean(axis=0)
cov = np.cov(Xc, rowvar=False)
print("covariance shape:", cov.shape, "symmetric:", np.allclose(cov, cov.T))

evals, evecs = np.linalg.eigh(cov)
order = np.argsort(evals)[::-1]
evals, evecs = evals[order], evecs[:, order]
print("top 5 eigenvalues:", evals[:5])

In [ ]:
explained = evals / evals.sum()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(range(1, 21), explained[:20])
axes[0].set_xlabel("component")
axes[0].set_ylabel("fraction of variance")
axes[0].set_title("Scree plot")
axes[1].plot(np.cumsum(explained))
axes[1].set_xlabel("number of components")
axes[1].set_ylabel("cumulative variance")
axes[1].set_title("Cumulative explained variance")
plt.show()

print("components for 90% variance:", int(np.searchsorted(np.cumsum(explained), 0.9)) + 1)

In [ ]:
Z = Xc @ evecs[:, :2]
codes = pd.factorize(df["prognosis"])[0]
plt.figure(figsize=(7, 5))
plt.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="nipy_spectral", s=8, alpha=0.6)
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.title("Symptom data projected onto top 2 eigenvectors")
plt.show()

top = np.argsort(np.abs(evecs[:, 0]))[::-1][:8]
print([(features[i], round(evecs[i, 0], 3)) for i in top])

In [ ]:
k = 20
recon = Xc @ evecs[:, :k] @ evecs[:, :k].T
err = np.linalg.norm(Xc - recon) / np.linalg.norm(Xc)
print(f"relative reconstruction error with {k} components: {err:.3f}")

U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
print("singular values^2/(n-1) match eigenvalues:", np.allclose(S[:5] ** 2 / (len(Xc) - 1), evals[:5]))

# Part 2: Calculus

## 2.1 Derivatives

The derivative of $f$ at $x$ is the slope of the tangent line: how much the output changes per tiny change in the input.

$f'(x) = \lim_{h\to 0} \frac{f(x+h)-f(x)}{h}$

In [ ]:
f = lambda x: x ** 3 - 2 * x
df_dx = lambda x: 3 * x ** 2 - 2

x0 = 1.5
for h in [1.0, 0.1, 0.01, 0.001, 1e-6]:
    est = (f(x0 + h) - f(x0)) / h
    print(f"h = {h:<9} estimate = {est:.6f}   exact = {df_dx(x0):.6f}")

In [ ]:
xs = np.linspace(-2, 2, 300)
plt.plot(xs, f(xs), label="f(x)")
tangent = f(x0) + df_dx(x0) * (xs - x0)
plt.plot(xs, tangent, "--", label="tangent at x = 1.5")
plt.scatter([x0], [f(x0)], color="red", zorder=5)
plt.ylim(-4, 5)
plt.legend()
plt.title("Derivative = slope of the tangent")
plt.show()

In [ ]:
def numerical_derivative(fn, x, h=1e-5):
    return (fn(x + h) - fn(x - h)) / (2 * h)


sigmoid = lambda z: 1 / (1 + np.exp(-z))
d_sigmoid = lambda z: sigmoid(z) * (1 - sigmoid(z))
tanh_grad = lambda z: 1 - np.tanh(z) ** 2
relu_grad = lambda z: (z > 0).astype(float)

z = np.linspace(-6, 6, 400)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(z, sigmoid(z), label="sigmoid")
axes[0].plot(z, np.tanh(z), label="tanh")
axes[0].plot(z, np.maximum(0, z), label="relu")
axes[0].legend()
axes[0].set_title("activations")
axes[1].plot(z, d_sigmoid(z), label="sigmoid'")
axes[1].plot(z, tanh_grad(z), label="tanh'")
axes[1].plot(z, relu_grad(z), label="relu'")
axes[1].legend()
axes[1].set_title("derivatives")
plt.show()

print("max sigmoid derivative:", d_sigmoid(z).max())
print("check at 0.7:", d_sigmoid(0.7), numerical_derivative(sigmoid, 0.7))

The sigmoid derivative never exceeds 0.25 and is nearly zero for large $|z|$. Multiplying many such factors through layers is the origin of the vanishing-gradient problem.

## 2.2 Partial derivatives and the gradient

With several inputs, a partial derivative measures the slope along one axis while the others are held fixed. Stacking all partials gives the **gradient**, a vector that points in the direction of steepest increase.

$\nabla f = \left(\frac{\partial f}{\partial x}, \frac{\partial f}{\partial y}\right)$

In [ ]:
def g(p):
    x, y = p
    return x ** 2 + 3 * y ** 2


def grad_g(p):
    x, y = p
    return np.array([2 * x, 6 * y])


def numerical_gradient(fn, p, h=1e-5):
    p = np.asarray(p, dtype=float)
    grad = np.zeros_like(p)
    for i in range(len(p)):
        step = np.zeros_like(p)
        step[i] = h
        grad[i] = (fn(p + step) - fn(p - step)) / (2 * h)
    return grad


p = np.array([1.0, 2.0])
print("analytic:", grad_g(p))
print("numeric: ", numerical_gradient(g, p))

In [ ]:
xs, ys = np.meshgrid(np.linspace(-3, 3, 100), np.linspace(-3, 3, 100))
zs = xs ** 2 + 3 * ys ** 2

plt.figure(figsize=(6, 5))
plt.contour(xs, ys, zs, levels=15, cmap="viridis")
qx, qy = np.meshgrid(np.linspace(-2.5, 2.5, 8), np.linspace(-2.5, 2.5, 8))
plt.quiver(qx, qy, 2 * qx, 6 * qy, color="tab:red", alpha=0.7)
plt.gca().set_aspect("equal")
plt.title("Gradient arrows are perpendicular to contour lines and point uphill")
plt.show()

## 2.3 Gradient descent

To minimise a function, step opposite the gradient: $\theta \leftarrow \theta - \eta\,\nabla f(\theta)$. The learning rate $\eta$ sets the step size.

In [ ]:
def descend(grad_fn, start, lr, steps):
    path = [np.array(start, dtype=float)]
    for _ in range(steps):
        path.append(path[-1] - lr * grad_fn(path[-1]))
    return np.array(path)


fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, lr in zip(axes, [0.02, 0.15, 0.36]):
    path = descend(grad_g, [-2.5, 2.0], lr, 25)
    ax.contour(xs, ys, zs, levels=15, cmap="viridis", alpha=0.6)
    ax.plot(path[:, 0], path[:, 1], "o-", color="tab:red", markersize=3)
    ax.set_title(f"lr = {lr}, final loss = {g(path[-1]):.3f}")
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
plt.show()

In [ ]:
for lr in [0.01, 0.1, 0.3, 0.34, 0.4]:
    path = descend(grad_g, [-2.5, 2.0], lr, 40)
    print(f"lr = {lr:<5} final loss = {g(path[-1]):.4g}")

Small learning rates crawl, moderate ones converge quickly, and ones that are too large overshoot and diverge. Here the curvature along $y$ is three times that along $x$, so the stable range is capped by the steeper direction. The stable range is tied to the largest eigenvalue of the Hessian, which links back to Part 1.

In [ ]:
H = np.array([[2.0, 0.0], [0.0, 6.0]])
print("Hessian eigenvalues:", np.linalg.eigvalsh(H))
print("stability limit for lr:", 2 / np.linalg.eigvalsh(H).max())

## 2.4 The chain rule

If $y = f(u)$ and $u = g(x)$, then

$\frac{dy}{dx} = \frac{dy}{du}\cdot\frac{du}{dx}$

Rates multiply along the chain. Backpropagation is this rule applied systematically through a network.

In [ ]:
u_fn = lambda x: 3 * x + 1
y_fn = lambda u: u ** 2
composite = lambda x: y_fn(u_fn(x))

x0 = 2.0
du_dx = 3.0
dy_du = 2 * u_fn(x0)
print("chain rule:", dy_du * du_dx)
print("numerical: ", numerical_derivative(composite, x0))

In [ ]:
def forward(x):
    a = 2 * x
    b = np.sin(a)
    c = b ** 2
    return a, b, c


x0 = 0.7
a, b, c = forward(x0)
dc_db = 2 * b
db_da = np.cos(a)
da_dx = 2.0
print("chain rule:", dc_db * db_da * da_dx)
print("numerical: ", numerical_derivative(lambda x: forward(x)[2], x0))

### Chain rule with several inputs

When a variable influences the output through multiple paths, add the contributions from each path. This is why gradients accumulate when a weight is reused or a neuron feeds several downstream units.

In [ ]:
def h_fn(x):
    p = x ** 2
    q = np.exp(x)
    return p * q


x0 = 0.5
dp, dq = 2 * x0, np.exp(x0)
p, q = x0 ** 2, np.exp(x0)
print("sum over paths:", q * dp + p * dq)
print("numerical:     ", numerical_derivative(h_fn, x0))

## 2.5 Backpropagation by hand

A tiny network: $z = w x + b$, $a = \sigma(z)$, loss $L = (a - t)^2$. The chain rule gives

$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial a}\cdot\frac{\partial a}{\partial z}\cdot\frac{\partial z}{\partial w}$

In [ ]:
x, t = 1.5, 1.0
w, b = 0.4, -0.2

z = w * x + b
a = sigmoid(z)
L = (a - t) ** 2

dL_da = 2 * (a - t)
da_dz = a * (1 - a)
dz_dw = x
dz_db = 1.0

dL_dw = dL_da * da_dz * dz_dw
dL_db = dL_da * da_dz * dz_db

def loss_wb(w_, b_):
    return (sigmoid(w_ * x + b_) - t) ** 2

print("backprop dL/dw:", dL_dw, " numeric:", numerical_derivative(lambda w_: loss_wb(w_, b), w))
print("backprop dL/db:", dL_db, " numeric:", numerical_derivative(lambda b_: loss_wb(w, b_), b))

### A two-layer network in matrix form

Gradients flow backward as matrix products. For $h = \mathrm{relu}(W_1 x)$, $\hat y = W_2 h$, and $L = \tfrac12\|\hat y - y\|^2$:

In [ ]:
x = rng.normal(size=(4,))
y = rng.normal(size=(3,))
W1 = rng.normal(size=(5, 4)) * 0.5
W2 = rng.normal(size=(3, 5)) * 0.5


def loss_fn(W1, W2):
    h = np.maximum(0, W1 @ x)
    y_hat = W2 @ h
    return 0.5 * np.sum((y_hat - y) ** 2)


pre = W1 @ x
h = np.maximum(0, pre)
y_hat = W2 @ h

d_yhat = y_hat - y
dW2 = np.outer(d_yhat, h)
d_h = W2.T @ d_yhat
d_pre = d_h * (pre > 0)
dW1 = np.outer(d_pre, x)

print("dW2 shape:", dW2.shape, " dW1 shape:", dW1.shape)

In [ ]:
def numeric_grad_matrix(fn, M, h=1e-5):
    grad = np.zeros_like(M)
    for idx in np.ndindex(*M.shape):
        orig = M[idx]
        M[idx] = orig + h
        up = fn()
        M[idx] = orig - h
        down = fn()
        M[idx] = orig
        grad[idx] = (up - down) / (2 * h)
    return grad


num_dW1 = numeric_grad_matrix(lambda: loss_fn(W1, W2), W1)
num_dW2 = numeric_grad_matrix(lambda: loss_fn(W1, W2), W2)
print("max |dW1 - numeric|:", np.abs(dW1 - num_dW1).max())
print("max |dW2 - numeric|:", np.abs(dW2 - num_dW2).max())

The gradient check agrees to within numerical precision, which is the standard way to verify a hand-written backward pass. Note how `W2.T` carries the error backward: the transpose of the forward matrix routes the loss signal to earlier layers.

## 2.6 Putting it together: training a network with hand-derived gradients

A small two-layer classifier on the symptom data, using only numpy, the matrix forms above, softmax cross-entropy, and gradient descent.

In [ ]:
classes, y_idx = np.unique(df["prognosis"].str.strip(), return_inverse=True)
n_classes = len(classes)
perm = rng.permutation(len(X))
split = int(0.8 * len(X))
tr, va = perm[:split], perm[split:]
Xtr, ytr, Xva, yva = X[tr], y_idx[tr], X[va], y_idx[va]

H = 32
W1 = rng.normal(size=(X.shape[1], H)) * np.sqrt(2 / X.shape[1])
b1 = np.zeros(H)
W2 = rng.normal(size=(H, n_classes)) * np.sqrt(2 / H)
b2 = np.zeros(n_classes)


def softmax(logits):
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def forward_pass(Xb):
    pre = Xb @ W1 + b1
    h = np.maximum(0, pre)
    return pre, h, softmax(h @ W2 + b2)


lr = 0.5
losses = []
for epoch in range(1, 101):
    pre, h, probs = forward_pass(Xtr)
    n = len(Xtr)
    loss = -np.log(probs[np.arange(n), ytr] + 1e-12).mean()
    losses.append(loss)

    d_logits = probs.copy()
    d_logits[np.arange(n), ytr] -= 1
    d_logits /= n
    dW2 = h.T @ d_logits
    db2 = d_logits.sum(axis=0)
    d_h = d_logits @ W2.T
    d_pre = d_h * (pre > 0)
    dW1 = Xtr.T @ d_pre
    db1 = d_pre.sum(axis=0)

    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    if epoch % 20 == 0 or epoch == 1:
        val_acc = (forward_pass(Xva)[2].argmax(1) == yva).mean()
        print(f"epoch {epoch:3d} | loss {loss:.4f} | val acc {val_acc:.3f}")

plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("training loss")
plt.title("Loss from hand-written backprop")
plt.show()

## Summary

| Idea | Intuition | Where it shows up |
|---|---|---|
| Dot product | Alignment between two vectors | Neurons, attention scores, cosine similarity |
| Matrix | A linear transformation of space | Layers, embeddings, batches |
| Eigenvector / eigenvalue | Directions a matrix only stretches | PCA, stability of repeated multiplication |
| Derivative | Local slope | Sensitivity of loss to one parameter |
| Gradient | Direction of steepest ascent | Gradient descent updates |
| Chain rule | Rates multiply along a chain | Backpropagation |